<a href="https://colab.research.google.com/github/alirezaataei1375-eng/Convergence-rate-QLLG-Bear-2/blob/main/Classical_Limit_QLLG_RBM%2C_bear_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# ==========================================================
# Imports
# ==========================================================
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gammaln



# ==========================================================
# Classical LLG
# ==========================================================

gamma = 1.0
kappa = 0.3
hbar = 1.0

gamma_eff = 1.0/(1.0 + kappa**2)

def normalize(v):

    n = np.linalg.norm(v)

    if n < 1e-12:
        return v

    return v/n


def llg_rhs(n, B):

    return (
        -gamma_eff*np.cross(n,B)
        -
        kappa*gamma_eff*np.cross(
            n,
            np.cross(n,B)
        )
    )
def llg_chain_rhs(
    n,
    Bext,
    S,
    J
):

    N = len(n)

    left = np.roll(n, 1, axis=0)

    right = np.roll(n, -1, axis=0)

    Beff = (
    gamma * Bext/(S*hbar)
    -
    (J/(S*hbar)**2)*(left + right)
)
    rhs = (
        -gamma_eff*np.cross(n, Beff)
        -
        kappa*gamma_eff*np.cross(
            n,
            np.cross(n, Beff)
        )
    )

    return rhs

    """
    rhs = np.zeros_like(n)

    for i in range(N):

        left = (i - 1) % N
        right = (i + 1) % N

        Beff = (
            gamma * Bext / (S*hbar)
            -
            (J/(S*hbar)**2)
            *
            (
                n[left]
                +
                n[right]
            )
        )

        rhs[i] = llg_rhs(
            n[i],
            Beff
        )

    return rhs
    """
def rk4_step_chain(
    n,
    Bext,
    S,
    J,
    dt
):

    k1 = llg_chain_rhs(
        n,
        Bext,
        S,
        J
    )

    k2 = llg_chain_rhs(
        n + 0.5*dt*k1,
        Bext,
        S,
        J
    )

    k3 = llg_chain_rhs(
        n + 0.5*dt*k2,
        Bext,
        S,
        J
    )

    k4 = llg_chain_rhs(
        n + dt*k3,
        Bext,
        S,
        J
    )

    n_new = n + dt*(
        k1
        + 2*k2
        + 2*k3
        + k4
    )/6

    for i in range(len(n_new)):
        n_new[i] = normalize(
            n_new[i]
        )

    return n_new



def angle_between(v1,v2):

    n1 = normalize(v1)
    n2 = normalize(v2)

    return np.arccos(
        np.clip(
            np.dot(n1,n2),
            -1.0,
            1.0
        )
    )

# ==========================================================
# Coherent state
# ==========================================================
def coherent_log_amplitude(
    m,
    S,
    theta,
    phi
):

    n = int(round(S + m))

    logbinom = 0.5 * (
        gammaln(2*S + 1)
        - gammaln(n + 1)
        - gammaln(2*S - n + 1)
    )

    return (
        logbinom
        +
        (S+m)*np.log(
            np.cos(theta/2)
        )
        +
        (S-m)*np.log(
            np.sin(theta/2)
        )
        -
        1j*m*phi
    )

# ==========================================================
# Spin-S Sparse RBM
# ==========================================================
class SpinSRBM:

    def __init__(
        self,
        N,
        S,
        alpha=1,
        connectivity=3,
        scale=0.01,
        seed=1234
    ):

        self.N = N
        self.S = S

        #self.local_states = np.arange(-S, S + 1)

        self.local_states = np.linspace(-S,S,int(2*S + 1))

        self.n_local = int(2*S + 1)

        self.A = np.zeros((N, self.n_local),dtype=complex)

        self.Nh = alpha * N
        self.z = connectivity

        rng = np.random.default_rng(seed)

        #self.a = scale * (
        #    rng.normal(size=N)
        #    + 1j * rng.normal(size=N)
        #)

        self.b = scale * (
            rng.normal(size=self.Nh)
            + 1j * rng.normal(size=self.Nh)
        )

        self.connections = np.zeros(
            (self.Nh, self.z),
            dtype=int
        )

        self.W = np.zeros(
            (self.Nh, self.z),
            dtype=complex
        )

        for j in range(self.Nh):

            sites = rng.choice(
                N,
                connectivity,
                replace=False
            )

            self.connections[j] = sites

            self.W[j] = scale * (
                rng.normal(size=self.z)
                + 1j * rng.normal(size=self.z)
            )

    # --------------------------------------------------
    # Number of trainable parameters
    # --------------------------------------------------
    def n_parameters(self):

        return (
            self.N * self.n_local
            + self.Nh
            + self.Nh * self.z
        )

    # --------------------------------------------------
    # Flatten parameters
    # --------------------------------------------------
    def parameters(self):

        theta = []

        theta.extend(self.A.flatten())
        theta.extend(self.b)

        for w in self.W:
            theta.extend(w)

        return np.asarray(theta, dtype=complex)

    # --------------------------------------------------
    # Restore parameters
    # --------------------------------------------------
    def set_parameters(self, theta):

        theta = np.asarray(theta, dtype=complex)

        p = 0

        nA = self.N*self.n_local

        self.A = theta[p:p+nA].reshape(self.N,self.n_local)

        p += nA

        self.b = theta[p:p+self.Nh]
        p += self.Nh

        W_new = []

        for j in range(self.Nh):

            W_new.append(
                theta[p:p+self.z]
            )

            p += self.z

        self.W = np.array(W_new)

    # --------------------------------------------------
    # Log wavefunction
    # --------------------------------------------------
    def log_psi(self, config):

        x = (self.b+ np.sum(self.W* config[self.connections],axis=1))

        idx = np.rint(config + self.S).astype(int)

        visible = np.sum(self.A[np.arange(self.N), idx])

        """
        visible = 0.0 + 0.0j

        for i, m in enumerate(config):

            idx = int(round(m + self.S))

            visible += self.A[i, idx]
        """
        return ( visible +  np.sum( np.log( 2.0*np.cosh(x))))

    # --------------------------------------------------
    # Initializing State
    # --------------------------------------------------

    def initialize_random_product_coherent(
    self,
    seed=None
):
        """
        Initialize a product of independently random spin-coherent states.

        Each site has a random direction uniformly distributed on the sphere.
        """

        rng = np.random.default_rng(seed)

        # Random directions uniformly distributed on S^2
        cos_theta = rng.uniform(-1.0, 1.0, size=self.N)
        theta = np.arccos(cos_theta)

        phi = rng.uniform(0.0, 2.0*np.pi, size=self.N)

        # Reset hidden parameters
        self.b[:] = 0.0
        self.W[:] = 0.0

        # Initialize each site with its own coherent state
        for i in range(self.N):

            for k, m in enumerate(self.local_states):

                self.A[i, k] = coherent_log_amplitude(
                m,
                self.S,
                theta[i],
                phi[i]
            )

        return theta, phi



# ==========================================================
# Spin-S Metropolis Sampler
# ==========================================================
class SpinSMetropolisSampler:

    def __init__(self, rbm, seed=12345):

        self.rbm = rbm

        self.N = rbm.N
        self.S = rbm.S

        self.rng = np.random.default_rng(seed)

        self.config = self.rng.choice(
            rbm.local_states,
            size=self.N
        )

        self.logpsi_current = (
            rbm.log_psi(self.config)
        )

    # --------------------------------------------------
    # One Metropolis step
    # --------------------------------------------------
    def step(self):

        i = self.rng.integers(self.N)

        delta = self.rng.choice([-1, 1])

        candidate = self.config[i] + delta

        if (
            candidate < -self.S
            or candidate > self.S
        ):
            return False

        new_config = self.config.copy()

        new_config[i] = candidate

        new_logpsi = (
            self.rbm.log_psi(new_config)
        )

        ratio = np.exp(
            new_logpsi
            - self.logpsi_current
        )

        prob = np.abs(ratio)**2

        if self.rng.random() < min(1.0, prob):

            self.config = new_config
            self.logpsi_current = new_logpsi

            return True

        return False

    # --------------------------------------------------
    # Thermalization
    # --------------------------------------------------
    def thermalize(self, sweeps=100):

        for _ in range(
            sweeps * self.N
        ):
            self.step()

    # --------------------------------------------------
    # Generate samples
    # --------------------------------------------------
    def samples(
        self,
        nsamples,
        decorrelation=5
    ):

        configs = []

        for _ in range(nsamples):

            for _ in range(
                decorrelation*self.N
            ):
                self.step()

            configs.append(
                self.config.copy()
            )

        return np.array(configs)




# ==========================================================
# Spin-S Heisenberg Hamiltonian
# ==========================================================
class SpinSHeisenbergHamiltonian:

    def __init__(
        self,
        S,
        J=1.0,
        h=0.0,
        periodic=True
    ):

        self.S = S
        self.J = J/(self.S**2)
        self.h = h/self.S
        self.periodic = periodic

    # --------------------------------------------------
    # Local energy
    # --------------------------------------------------
    def local_energy(
        self,
        rbm,
        config
    ):

        N = rbm.N
        S = self.S

        E = 0.0 + 0.0j

        logpsi0 = rbm.log_psi(config)


        bonds = (
            N
            if self.periodic
            else N-1
        )

        for i in range(bonds):

            j = (i + 1) % N

            mi = config[i]
            mj = config[j]

            # -----------------------------------------
            # Sz Sz
            # -----------------------------------------
            E += self.J * mi * mj

            # -----------------------------------------
            # S+ S-
            # -----------------------------------------
            if (
                mi < S
                and
                mj > -S
            ):

                cfg = config.copy()

                cfg[i] += 1
                cfg[j] -= 1

                coeff = np.sqrt(
                    S*(S+1)
                    - mi*(mi+1)
                )

                coeff *= np.sqrt(
                    S*(S+1)
                    - mj*(mj-1)
                )
                """
                ratio = np.exp(
                    rbm.log_psi(cfg)
                    -
                    rbm.log_psi(config)
                )
                """
                ratio = np.exp(rbm.log_psi(cfg)- logpsi0)

                E += (
                    0.5
                    * self.J
                    * coeff
                    * ratio
                )

            # -----------------------------------------
            # S- S+
            # -----------------------------------------
            if (
                mi > -S
                and
                mj < S
            ):

                cfg = config.copy()

                cfg[i] -= 1
                cfg[j] += 1

                coeff = np.sqrt(
                    S*(S+1)
                    - mi*(mi-1)
                )

                coeff *= np.sqrt(
                    S*(S+1)
                    - mj*(mj+1)
                )

                ratio = np.exp(
                    rbm.log_psi(cfg)
                    -
                    logpsi0
                )

                E += (
                    0.5
                    * self.J
                    * coeff
                    * ratio
                )

        # longitudinal field

        E += (
            - self.h
            * np.sum(config)
        )

        return E



# ==========================================================
# Derivatives
# ==========================================================

def derivatives(rbm, config):

    dA = np.zeros((rbm.N, rbm.n_local),dtype=complex)

    for i, m in enumerate(config):

        idx = int(round(m + rbm.S))

        dA[i, idx] = 1.0

    db = np.zeros(
        rbm.Nh,
        dtype=complex
    )

    dW = []

    for j in range(rbm.Nh):

        sites = rbm.connections[j]

        x = (
            rbm.b[j]
            +
            np.dot(
                rbm.W[j],
                config[sites]
            )
        )

        t = np.tanh(x)

        db[j] = t

        dW.append(
            config[sites] * t
        )

    return dA, db, dW


def flatten_derivatives(rbm, config):

    dA, db, dW = derivatives(
        rbm,
        config
    )

    params = []

    params.extend(dA.flatten())
    params.extend(db)

    for w in dW:
        params.extend(w)

    return np.asarray(
        params,
        dtype=complex
    )



# ==========================================================
# QLLG Simulation
# ==========================================================


def local_energy_qllg(
    ham,
    rbm,
    config,
    kappa
):

    coef = (
        (1.0 - 1j*kappa)
        /
        (1.0 + kappa**2)
    )

    return (
        coef
        *
        ham.local_energy(
            rbm,
            config
        )
    )


def build_qllg_tdvp(
    rbm,
    sampler,
    ham,
    nsamples,
    kappa
):

    configs = sampler.samples(
        nsamples,
        decorrelation=2
    )

    O_list = []
    E_list = []

    for config in configs:

        O = flatten_derivatives(
            rbm,
            config
        )

        E = local_energy_qllg(
            ham,
            rbm,
            config,
            kappa
        )

        O_list.append(O)
        E_list.append(E)

    O = np.asarray(O_list)

    E = np.asarray(E_list)

    O_mean = np.mean(
        O,
        axis=0
    )

    E_mean = np.mean(E)

    Oc = O - O_mean

    Ec = E - E_mean

    Smat = (
        Oc.conj().T
        @
        Oc
    ) / nsamples

    Fvec = (
        Oc.conj().T
        @
        Ec
    ) / nsamples

    return (
    Smat,
    Fvec,
    E_mean,
    configs
)



def solve_qllg(
    Smat,
    Fvec,
    reg=1e-5
):

    Sreg = (
        Smat
        +
        Smat.conj().T
    ) / 2

    vals, vecs = np.linalg.eigh(
        Sreg
    )

    vals = np.maximum(
        vals,
        1e-10
    )

    Sinv = (
        vecs
        @
        np.diag(
            1.0/(vals + reg)
        )
        @
        vecs.conj().T
    )

    theta_dot = (
        Sinv
        @
        (-1j * Fvec)
    )

    return theta_dot




# ==========================================================
# Spin Vectors
# ==========================================================

def local_Splus(
    rbm,
    config,
    site,
    S
):

    m = config[site]

    if m >= S:
        return 0.0

    cfg = config.copy()
    cfg[site] += 1

    coeff = np.sqrt(
        S*(S+1)
        -
        m*(m+1)
    )

    ratio = np.exp(
        rbm.log_psi(cfg)
        -
        rbm.log_psi(config)
    )

    return coeff * ratio

def local_Sminus(
    rbm,
    config,
    site,
    S
):

    m = config[site]

    if m <= -S:
        return 0.0

    cfg = config.copy()
    cfg[site] -= 1

    coeff = np.sqrt(
        S*(S+1)
        -
        m*(m-1)
    )

    ratio = np.exp(
        rbm.log_psi(cfg)
        -
        rbm.log_psi(config)
    )

    return coeff * ratio



def connected_correlation_xyz(
    rbm,
    configs,
    i,
    j,
    S
):

    nsamples = len(configs)

    sx_i = sy_i = sz_i = 0.0
    sx_j = sy_j = sz_j = 0.0

    sxsx = 0.0
    sysy = 0.0
    szsz = 0.0

    for cfg in configs:

        # spin i

        sp_i = local_Splus(
            rbm,
            cfg,
            i,
            S
        )

        sm_i = local_Sminus(
            rbm,
            cfg,
            i,
            S
        )

        sxi = np.real(
            0.5*(sp_i + sm_i)
        )

        syi = np.real(
            (sp_i - sm_i)/(2j)
        )

        szi = cfg[i]

        # spin j

        sp_j = local_Splus(
            rbm,
            cfg,
            j,
            S
        )

        sm_j = local_Sminus(
            rbm,
            cfg,
            j,
            S
        )

        sxj = np.real(
            0.5*(sp_j + sm_j)
        )

        syj = np.real(
            (sp_j - sm_j)/(2j)
        )

        szj = cfg[j]

        sx_i += sxi
        sy_i += syi
        sz_i += szi

        sx_j += sxj
        sy_j += syj
        sz_j += szj

        sxsx += sxi*sxj
        sysy += syi*syj
        szsz += szi*szj

    sx_i /= nsamples
    sy_i /= nsamples
    sz_i /= nsamples

    sx_j /= nsamples
    sy_j /= nsamples
    sz_j /= nsamples

    sxsx /= nsamples
    sysy /= nsamples
    szsz /= nsamples

    Cxx = sxsx - sx_i*sx_j
    Cyy = sysy - sy_i*sy_j
    Czz = szsz - sz_i*sz_j

    return Cxx, Cyy, Czz



def local_spin_vector(
    rbm,
    sampler,
    site,
    S,
    samples=1000
):

    configs = sampler.samples(
        samples,
        decorrelation=5
    )

    sx = 0.0 + 0.0j
    sy = 0.0 + 0.0j
    sz = 0.0

    for cfg in configs:

        sp = local_Splus(
            rbm,
            cfg,
            site,
            S
        )

        sm = local_Sminus(
            rbm,
            cfg,
            site,
            S
        )

        sx += 0.5*(sp + sm)

        sy += (
            (sp - sm)/(2j)
        )

        sz += cfg[site]

    sx /= samples
    sy /= samples
    sz /= samples

    return np.array([
        np.real(sx),
        np.real(sy),
        np.real(sz)
    ])


def length_defect(
    spin_vector,
    S
):

    return np.abs(
        1.0
        -
        np.linalg.norm(spin_vector)/S
    )

def local_spin_vector_from_configs(
    rbm,
    configs,
    site,
    S
):

    sx = 0.0 + 0.0j
    sy = 0.0 + 0.0j
    sz = 0.0

    nsamples = len(configs)

    for cfg in configs:

        sp = local_Splus(
            rbm,
            cfg,
            site,
            S
        )

        sm = local_Sminus(
            rbm,
            cfg,
            site,
            S
        )

        sx += 0.5*(sp + sm)

        sy += (sp - sm)/(2j)

        sz += cfg[site]

    sx /= nsamples
    sy /= nsamples
    sz /= nsamples

    return np.array([
        np.real(sx),
        np.real(sy),
        np.real(sz)
    ])



def get_all_spin_vectors(
    rbm,
    sampler,
    S,
    samples=100
):

    configs = sampler.samples(
        samples,
        decorrelation=5
    )

    spins = []

    for site in range(rbm.N):

        spins.append(
            local_spin_vector_from_configs(
                rbm,
                configs,
                site,
                S
            )
        )

    return np.array(spins)


def get_all_spin_vectors_from_configs(
    rbm,
    configs,
    S
):

    spins = []

    for site in range(rbm.N):

        spins.append(
            local_spin_vector_from_configs(
                rbm,
                configs,
                site,
                S
            )
        )

    return np.array(spins)


# ==========================================================
# FULL DIAGNOSTIC TEST
# ==========================================================





def run_qllg_defect(
    rbm,
    sampler,
    ham,
    N,
    S,
    steps,
    dt,
    kappa,
    samples=300,
    reg=1e-3
):

    mean_defect = 0.0

    mean_angle_list = []

    mean_length_list = []

    mean_corr_list = []

    Bext = np.array([1/3,2/3,2/3])

    spins = get_all_spin_vectors(rbm,sampler,S,samples=100)

    n_classical = np.array([normalize(v)for v in spins])

    for step in range(steps):

        Smat, Fvec, _, configs = build_qllg_tdvp(
            rbm,
            sampler,
            ham,
            samples,
            kappa
        )

        theta_dot = solve_qllg(
            Smat,
            Fvec,
            reg=reg
        )

        theta = rbm.parameters()

        rbm.set_parameters(
            theta + dt*theta_dot
        )



        spins_q = get_all_spin_vectors_from_configs(rbm,configs,S)


        n_classical = rk4_step_chain(n_classical,Bext,S,J=1.0,dt=dt)


        angles = []

        for i in range(N):

            angles.append(angle_between(spins_q[i],n_classical[i]))

        mean_angle_list.append(np.mean(angles))

        lengths = []

        for i in range(N):

            lengths.append(abs(1-np.linalg.norm(spins_q[i])/S))

        mean_length_list.append(np.mean(lengths))



        corrs = []

        for i in range(N):

            j = (i+1) % N

            Cxx, Cyy, Czz = connected_correlation_xyz(rbm,configs,i,j,S)

            Cnorm = np.sqrt(Cxx**2 +Cyy**2 +Czz**2) / S**2

            corrs.append(Cnorm)

        mean_corr_list.append(np.mean(corrs))





    return {"mean_angle": np.array(mean_angle_list), "mean_length": np.array(mean_length_list), "mean_corr": np.array(mean_corr_list)}



N = 20
S=13.5
rbm = SpinSRBM(
        N=N,
        S=S,
        alpha=1,
        connectivity=2
    )


theta, phi = rbm.initialize_random_product_coherent(
    seed=1234
)
sampler = SpinSMetropolisSampler(rbm)

sampler.thermalize(100)

v = local_spin_vector(
    rbm,
    sampler,
    0,
    S,
    samples=400
)

print(
    np.linalg.norm(v)/S
)




S_values = np.arange(
    0.5,
    40.5,
    0.5
)



N_values = [20,40]

results = {}

for N in N_values:

    print("\n")
    print("="*60)
    print(f"Running N = {N}")
    print("="*60)

    max_angles = []
    max_lengths = []
    max_corrs = []

    for S in S_values:

        print(f"Running N={N}, S={S}")

        rbm = SpinSRBM(
            N=N,
            S=S,
            alpha=1,
            connectivity=2
        )

        theta, phi = rbm.initialize_random_product_coherent( seed=1234)
        sampler = SpinSMetropolisSampler(rbm)

        sampler.thermalize(100)

        ham = SpinSHeisenbergHamiltonian(
            S=S,
            J=1.0,
            periodic=False
        )

        diag = run_qllg_defect(
            rbm,
            sampler,
            ham,
            N=N,
            S=S,
            steps=40,
            dt=5e-2,
            kappa=0.3,
            samples=500,
            reg=1e-5
        )

        angle = np.max(diag["mean_angle"])
        length = np.max(diag["mean_length"])
        corr = np.max(np.abs(diag["mean_corr"]))

        max_angles.append(angle)
        max_lengths.append(length)
        max_corrs.append(corr)

        print("  Max angle error      =", angle)
        print("  Max length defect (%)   =", length*100)
        print("  Max correlation   (%)   =", corr*100)

    results[N] = {
        "angles": np.array(max_angles),
        "lengths": np.array(max_lengths),
        "corrs": np.array(max_corrs)
    }


plt.figure(figsize=(7,5))

for N in N_values:
    plt.plot(
        S_values,
        results[N]["angles"],
        "o-",
        linewidth=2,
        label=f"N={N}"
    )

plt.xlabel("S",fontsize=16)
plt.ylabel("Angle error",fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend()
#plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


plt.figure(figsize=(7,5))

for N in N_values:
    plt.plot(
        S_values,
        results[N]["lengths"]*100,
        "o-",
        linewidth=2,
        label=f"N={N}"
    )

plt.xlabel("Spin S",fontsize=16)
plt.ylabel("Length error (%)",fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend()
#plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



plt.figure(figsize=(7,5))

for N in N_values:
    plt.plot(
        S_values,
        results[N]["corrs"]*100,
        "o-",
        linewidth=2,
        label=f"N={N}"
    )

plt.xlabel("Spin S",fontsize=16)
plt.ylabel("Correlation norm (%)",fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend()
#plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━ 24.1/28.5 MB 157.2 MB/s eta 0:00:01
ERROR: Operation cancelled by user


ModuleNotFoundError: No module named 'qutip'